# Checkpoint 1 — Machine Learning & Statistical Computing
## Case iFood — Previsão de Resposta a Campanha

**Turma:** 1TIAPR-2025 (2º semestre)

---

### Estrutura do Notebook

- **Parte 1 — Statistical Computing:** Teste Z para comparação de médias entre grupos (Response=0 vs Response=1)
- **Parte 2 — Machine Learning & Modelling:** Feature Engineering + Tuning de Hiperparâmetros + Métricas + Feature Importance
- **Parte 3 — Comparação:** Top 5 features do ML vs features significativas no Teste Z

### Contexto de Negócio
O objetivo é identificar quais clientes têm maior probabilidade de responder positivamente a uma campanha de marketing.  
Como **o orçamento de marketing é limitado**, precisamos ser precisos: melhor abordar poucos clientes certos do que muitos errados.  
Por isso, priorizamos **Precision** sem abrir mão de um **Recall aceitável**.

In [ ]:
# Instale bibliotecas caso necessário (descomente se precisar)
# !pip install pyarrow scipy

In [ ]:
# =============================================================
# IMPORTAÇÕES
# Importamos todas as bibliotecas necessárias para as 3 partes
# =============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Estatística — usada na Parte 1
from scipy import stats

# Pré-processamento e modelagem — Parte 2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("Bibliotecas importadas com sucesso!")

## 0. Leitura e Exploração dos Dados

Carregamos o dataset do iFood e fazemos uma exploração inicial para entender a estrutura dos dados.

In [ ]:
# =============================================================
# LEITURA DO DATASET
# Ajuste o caminho se necessário
# No Colab, faça upload do arquivo data.csv ou monte o Drive
# =============================================================

# Tentativa automática de leitura em diferentes caminhos
for path in ["data.csv", "/content/data.csv", "../data.csv"]:
    try:
        df_raw = pd.read_csv(path)
        print(f"Arquivo carregado de: {path}")
        break
    except FileNotFoundError:
        continue

print(f"\nShape do dataset: {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# =============================================================
# VISÃO GERAL DOS DADOS
# Checamos tipos, valores nulos e distribuição da variável-alvo
# =============================================================

print("=== Tipos de dados ===")
print(df_raw.dtypes)

print("\n=== Valores Nulos ===")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

print("\n=== Distribuição da variável Response ===")
print(df_raw['Response'].value_counts())
print("\nProporção:")
print(df_raw['Response'].value_counts(normalize=True).round(3))

## Feature Engineering — Criação de Variáveis

Antes de qualquer análise, criamos variáveis derivadas que fazem sentido de negócio:  
- **Age**: idade do cliente calculada a partir do ano de nascimento  
- **Days_Customer**: há quantos dias o cliente está cadastrado  
- **TotalMnt**: total gasto em todos os produtos  
- **TotalPurchases**: total de compras em todos os canais  
- **TotalCampaigns**: quantas campanhas anteriores o cliente aceitou  
- **HasChildren**: se o cliente tem filhos (kid ou teen)  
- **IsAlone**: se o cliente é solteiro/separado/viúvo

In [ ]:
# =============================================================
# FEATURE ENGINEERING
# Criamos novas variáveis derivadas das originais.
# O objetivo é capturar informações de negócio de forma mais direta.
# =============================================================

df = df_raw.copy()

# --- Variáveis de tempo ---
# Calculamos a idade do cliente (referência: 2024)
df['Age'] = 2024 - df['Year_Birth']

# Convertemos a data de cadastro e calculamos há quantos dias é cliente
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], errors='coerce')
ref_date = df['Dt_Customer'].max()
df['Days_Customer'] = (ref_date - df['Dt_Customer']).dt.days

# --- Variáveis de gastos ---
# Somamos todos os gastos em produtos para ter o total gasto
mnt_cols = [c for c in df.columns if c.startswith('Mnt')]
df['TotalMnt'] = df[mnt_cols].sum(axis=1)

# --- Variáveis de compras ---
# Somamos todas as compras por canal
purchase_cols = [c for c in df.columns if 'Purchases' in c]
df['TotalPurchases'] = df[purchase_cols].sum(axis=1)

# --- Histórico de campanhas ---
# Quantas campanhas anteriores o cliente aceitou?
cmp_cols = [c for c in df.columns if 'AcceptedCmp' in c]
df['TotalCampaigns'] = df[cmp_cols].sum(axis=1)

# --- Variáveis de perfil familiar ---
# Se tem algum filho ou adolescente em casa
df['HasChildren'] = ((df['Kidhome'] + df['Teenhome']) > 0).astype(int)

# Se mora sozinho (solteiro, separado ou viúvo)
alone_status = ['Single', 'Divorced', 'Widow', 'Alone', 'Absurd', 'YOLO']
df['IsAlone'] = df['Marital_Status'].isin(alone_status).astype(int)

print("Novas features criadas:")
new_features = ['Age', 'Days_Customer', 'TotalMnt', 'TotalPurchases', 'TotalCampaigns', 'HasChildren', 'IsAlone']
print(df[new_features].describe().round(2))

---
# PARTE 1 — Statistical Computing
## Teste Z para Comparação de Médias entre Grupos

### O que é o Teste Z?
O Teste Z é um teste estatístico usado para verificar se a **média de dois grupos é significativamente diferente**.  
Neste caso, comparamos os clientes que **responderam** à campanha (Response=1) com os que **não responderam** (Response=0).

### Como funciona?
- **H₀ (hipótese nula):** as médias dos dois grupos são iguais (não há diferença)  
- **H₁ (hipótese alternativa):** as médias são diferentes  
- Se o **p-valor < 0.05**, rejeitamos H₀ → a diferença é estatisticamente significativa  

### Por que o Teste Z?
Com mais de 2.200 registros, nossos grupos têm tamanho suficiente para o Teste Z ser válido (n > 30 em ambos os grupos).

In [ ]:
# =============================================================
# PARTE 1 — TESTE Z
# Separamos os clientes em dois grupos e comparamos as médias
# de cada feature numérica usando o Teste Z.
# =============================================================

# Separação dos grupos
grupo_0 = df[df['Response'] == 0]  # Não responderam
grupo_1 = df[df['Response'] == 1]  # Responderam

print(f"Grupo Response=0: {len(grupo_0)} clientes")
print(f"Grupo Response=1: {len(grupo_1)} clientes")

# Selecionamos as colunas numéricas para o teste
# Excluímos colunas sem variância ou que são identificadores
exclude_cols = ['ID', 'Year_Birth', 'Response', 'Z_CostContact', 'Z_Revenue', 'Dt_Customer']
num_cols_test = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_test = [c for c in num_cols_test if c not in exclude_cols]

print(f"\nNúmero de features numéricas a testar: {len(num_cols_test)}")

# =============================================================
# APLICAÇÃO DO TESTE Z
# Usamos scipy.stats.ranksums como aproximação do Teste Z
# para amostras grandes (equivalente ao Teste Z de Wilcoxon)
# Para amostras grandes, o Teste t_ind converge para o Teste Z
# =============================================================

resultados = []
ALPHA = 0.05  # Nível de significância

for col in num_cols_test:
    # Removemos NaN para cada coluna
    g0 = grupo_0[col].dropna()
    g1 = grupo_1[col].dropna()

    # Teste Z para duas amostras independentes (usando stats.ttest_ind com n grande = Teste Z)
    stat, pvalue = stats.ttest_ind(g0, g1, equal_var=False)  # Welch's t-test ≈ Z-test para n grande

    resultados.append({
        'Feature': col,
        'Média Response=0': round(g0.mean(), 3),
        'Média Response=1': round(g1.mean(), 3),
        'Diferença': round(g1.mean() - g0.mean(), 3),
        'Z-statistic': round(stat, 3),
        'p-valor': round(pvalue, 5),
        'Significativa': 'SIM ✓' if pvalue < ALPHA else 'NÃO'
    })

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('p-valor')

print("\n=== RESULTADOS DO TESTE Z ===")
print(df_resultados.to_string(index=False))

In [ ]:
# =============================================================
# FEATURES ESTATISTICAMENTE SIGNIFICATIVAS
# Filtramos apenas as features com p-valor < 0.05
# =============================================================

features_significativas = df_resultados[df_resultados['p-valor'] < ALPHA]['Feature'].tolist()

print(f"Features estatisticamente significativas (α = {ALPHA}):")
print(f"Total: {len(features_significativas)}\n")
for f in features_significativas:
    row = df_resultados[df_resultados['Feature'] == f].iloc[0]
    print(f"  {f}: média(0)={row['Média Response=0']}, média(1)={row['Média Response=1']}, p={row['p-valor']}")

In [ ]:
# =============================================================
# VISUALIZAÇÃO DOS RESULTADOS DO TESTE Z
# Gráfico de barras com as 15 features de menor p-valor
# =============================================================

top_features = df_resultados.head(15).copy()
colors = ['#e63946' if p < ALPHA else '#adb5bd' for p in top_features['p-valor']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Teste Z — Comparação de Médias entre Response=0 e Response=1', fontsize=14, fontweight='bold')

# Gráfico 1: p-valores
ax1 = axes[0]
bars = ax1.barh(top_features['Feature'], top_features['p-valor'], color=colors)
ax1.axvline(x=ALPHA, color='black', linestyle='--', label=f'α = {ALPHA}')
ax1.set_xlabel('p-valor')
ax1.set_title('p-valores (vermelho = significativo)')
ax1.legend()
ax1.invert_yaxis()

# Gráfico 2: Diferença de médias
ax2 = axes[1]
diff_colors = ['#2a9d8f' if d > 0 else '#e76f51' for d in top_features['Diferença']]
ax2.barh(top_features['Feature'], top_features['Diferença'], color=diff_colors)
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_xlabel('Diferença de Médias (Response=1 - Response=0)')
ax2.set_title('Diferença de Médias entre os Grupos')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('ztest_resultado.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico salvo como ztest_resultado.png")

---
# PARTE 2 — Machine Learning & Modelling
## Feature Engineering + Tuning + Avaliação + Feature Importance

### Estratégia
1. Preparar os dados com as features criadas na etapa anterior
2. Treinar modelos baseline (como no Part 1 original)
3. Melhorar com **tuning de hiperparâmetros** via GridSearchCV
4. Avaliar com as métricas corretas para o problema
5. Identificar as features mais importantes

### Métricas de Avaliação — Por que Precision e F1?
- **Precision:** dos clientes que prevemos como positivos, quantos realmente são? Importante pois temos orçamento limitado.
- **Recall:** dos clientes que realmente são positivos, quantos conseguimos capturar?
- **F1-Score:** média harmônica entre Precision e Recall — equilíbrio entre os dois
- **AUC-ROC:** capacidade geral do modelo de separar as classes

> **Justificativa:** Em campanhas de marketing com orçamento limitado, **Precision** é mais importante para não desperdiçar contatos. Usamos F1 como métrica principal de tuning por equilibrar Precision e Recall.

In [ ]:
# =============================================================
# PREPARAÇÃO DOS DADOS PARA MODELAGEM
# Definimos X (features) e y (alvo), e fazemos o split
# =============================================================

TARGET = 'Response'

# Removemos colunas que não devem entrar no modelo
cols_drop = ['ID', 'Year_Birth', 'Dt_Customer', TARGET]
X_raw = df.drop(columns=cols_drop)
y = df[TARGET].astype(int)

print(f"Shape de X: {X_raw.shape}")
print(f"Distribuição do target: {dict(y.value_counts())}")

# Split treino/teste com estratificação (mantém proporção da classe)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

print(f"\nTreino: {X_train.shape}, Teste: {X_test.shape}")

In [ ]:
# =============================================================
# PRÉ-PROCESSAMENTO
# Criamos pipelines separadas para modelos lineares (com scaler)
# e modelos de árvore (sem necessidade de normalização)
# =============================================================

# Identificamos colunas categóricas e numéricas
cat_cols = X_train.select_dtypes(include=['object', 'string', 'category', 'boolean']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

print(f"Features numéricas ({len(num_cols)}): {num_cols}")
print(f"Features categóricas ({len(cat_cols)}): {cat_cols}")

# Pipeline para modelos lineares (precisa de escalonamento)
numeric_pipe_log = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Preenche NaN com mediana
    ('scaler', StandardScaler())                    # Normaliza entre média=0 e desvio=1
])

# Pipeline para modelos de árvore (não precisa de escalonamento)
numeric_pipe_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))   # Só preenche NaN
])

# Pipeline para variáveis categóricas (converte texto em números)
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Preenche NaN com moda
    ('onehot', OneHotEncoder(handle_unknown='ignore'))      # One-Hot Encoding
])

# ColumnTransformer aplica pipelines diferentes para cada tipo de coluna
preprocessor_log = ColumnTransformer([
    ('num', numeric_pipe_log, num_cols),
    ('cat', categorical_pipe, cat_cols)
])

preprocessor_tree = ColumnTransformer([
    ('num', numeric_pipe_tree, num_cols),
    ('cat', categorical_pipe, cat_cols)
])

print("\nPré-processadores criados com sucesso!")

In [ ]:
# =============================================================
# FUNÇÃO DE AVALIAÇÃO
# Usamos esta função para avaliar qualquer modelo de forma padronizada
# =============================================================

def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """
    Treina o modelo e retorna as métricas de avaliação.
    Threshold é o ponto de corte para classificar como positivo (padrão: 0.5).
    """
    modelo.fit(X_tr, y_tr)

    # Probabilidades de pertencer à classe 1
    y_prob = modelo.predict_proba(X_te)[:, 1]
    # Classificação binária usando o threshold definido
    y_pred = (y_prob >= threshold).astype(int)

    metricas = {
        'Modelo': nome,
        'Threshold': threshold,
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_te, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_te, y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_te, y_prob), 4),
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
    }
    return metricas, modelo

print("Função de avaliação definida!")

In [ ]:
# =============================================================
# MODELOS BASELINE
# Treinamos os 3 modelos sem tuning para ter uma referência
# de desempenho inicial
# =============================================================

resultados_baseline = []
modelos_baseline = {}

# 1. Regressão Logística (baseline)
pipe_log = Pipeline([
    ('preprocessor', preprocessor_log),
    ('model', LogisticRegression(max_iter=1000, random_state=SEED))
])
met, mod = avaliar_modelo('LogReg Baseline', pipe_log, X_train, y_train, X_test, y_test)
resultados_baseline.append(met)
modelos_baseline['LogReg'] = mod

# 2. Decision Tree (baseline)
pipe_dt = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', DecisionTreeClassifier(random_state=SEED))
])
met, mod = avaliar_modelo('DTree Baseline', pipe_dt, X_train, y_train, X_test, y_test)
resultados_baseline.append(met)
modelos_baseline['DTree'] = mod

# 3. Random Forest (baseline)
pipe_rf = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', RandomForestClassifier(random_state=SEED))
])
met, mod = avaliar_modelo('RForest Baseline', pipe_rf, X_train, y_train, X_test, y_test)
resultados_baseline.append(met)
modelos_baseline['RForest'] = mod

df_baseline = pd.DataFrame(resultados_baseline)
print("=== RESULTADOS BASELINE ===")
print(df_baseline.to_string(index=False))

In [ ]:
# =============================================================
# TUNING DE HIPERPARÂMETROS — DECISION TREE
# Usamos GridSearchCV para encontrar os melhores hiperparâmetros
# da Decision Tree de forma sistemática.
#
# Hiperparâmetros testados:
#   - max_depth: profundidade máxima da árvore (controla complexidade)
#   - min_samples_split: mínimo de amostras para dividir um nó
#   - min_samples_leaf: mínimo de amostras em cada folha
# =============================================================

print("Iniciando tuning da Decision Tree... (pode demorar alguns segundos)")

pipe_dt_tune = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', DecisionTreeClassifier(random_state=SEED))
])

# Grade de parâmetros para busca
# Prefixo 'model__' indica que o parâmetro é do step 'model' na pipeline
param_grid_dt = {
    'model__max_depth': [3, 5, 7, 10, None],
    'model__min_samples_split': [2, 10, 20],
    'model__min_samples_leaf': [1, 5, 10]
}

# GridSearchCV testa todas as combinações com validação cruzada de 5 folds
# scoring='f1' = otimizamos pelo F1-Score
gs_dt = GridSearchCV(
    pipe_dt_tune,
    param_grid_dt,
    cv=5,
    scoring='f1',
    n_jobs=-1,    # Usa todos os núcleos disponíveis
    verbose=0
)
gs_dt.fit(X_train, y_train)

print(f"Melhores parâmetros: {gs_dt.best_params_}")
print(f"Melhor F1 no CV: {gs_dt.best_score_:.4f}")

In [ ]:
# =============================================================
# TUNING DE HIPERPARÂMETROS — RANDOM FOREST
# Random Forest é um ensemble de Decision Trees.
# Além dos parâmetros da árvore, controlamos:
#   - n_estimators: número de árvores no ensemble
#   - max_features: quantas features considerar em cada divisão
# =============================================================

print("Iniciando tuning do Random Forest... (pode demorar 1-2 minutos)")

pipe_rf_tune = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', RandomForestClassifier(random_state=SEED))
])

param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_leaf': [1, 5],
    'model__max_features': ['sqrt', 'log2']
}

gs_rf = GridSearchCV(
    pipe_rf_tune,
    param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
gs_rf.fit(X_train, y_train)

print(f"Melhores parâmetros: {gs_rf.best_params_}")
print(f"Melhor F1 no CV: {gs_rf.best_score_:.4f}")

In [ ]:
# =============================================================
# TUNING DE HIPERPARÂMETROS — REGRESSÃO LOGÍSTICA
# Hiperparâmetros testados:
#   - C: inverso da regularização (menor C = mais regularização)
#   - penalty: tipo de regularização (L1 ou L2)
# =============================================================

print("Iniciando tuning da Regressão Logística...")

pipe_log_tune = Pipeline([
    ('preprocessor', preprocessor_log),
    ('model', LogisticRegression(max_iter=1000, random_state=SEED))
])

param_grid_log = {
    'model__C': [0.01, 0.1, 1.0, 10.0],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'saga']
}

gs_log = GridSearchCV(
    pipe_log_tune,
    param_grid_log,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
gs_log.fit(X_train, y_train)

print(f"Melhores parâmetros: {gs_log.best_params_}")
print(f"Melhor F1 no CV: {gs_log.best_score_:.4f}")

In [ ]:
# =============================================================
# AVALIAÇÃO DOS MODELOS TUNADOS NO CONJUNTO DE TESTE
# Agora comparamos baseline vs modelos com tuning
# =============================================================

resultados_tunados = []

# Avaliamos os melhores estimadores encontrados pelo GridSearch
for nome, gs in [('LogReg Tunado', gs_log), ('DTree Tunado', gs_dt), ('RForest Tunado', gs_rf)]:
    best = gs.best_estimator_
    y_prob = best.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    resultados_tunados.append({
        'Modelo': nome,
        'Threshold': 0.5,
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, y_prob), 4),
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
    })

df_tunado = pd.DataFrame(resultados_tunados)

print("=== COMPARAÇÃO: BASELINE vs TUNADO ===")
print("\n--- BASELINE ---")
print(df_baseline.to_string(index=False))
print("\n--- TUNADO ---")
print(df_tunado.to_string(index=False))

In [ ]:
# =============================================================
# MATRIZ DE CONFUSÃO — MELHOR MODELO
# O Random Forest tunado geralmente é o melhor modelo.
# Visualizamos a matriz de confusão e a curva Precision-Recall.
# =============================================================

# Identificamos o melhor modelo pelo maior F1
best_nome = df_tunado.loc[df_tunado['F1'].idxmax(), 'Modelo']
print(f"Melhor modelo pelo F1: {best_nome}")

# Mapeamos o nome para o GridSearch correto
mapa_gs = {
    'LogReg Tunado': gs_log,
    'DTree Tunado': gs_dt,
    'RForest Tunado': gs_rf
}
best_model = mapa_gs[best_nome].best_estimator_

y_prob_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Avaliação do Melhor Modelo: {best_nome}', fontsize=13, fontweight='bold')

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Respondeu (0)', 'Respondeu (1)'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão')

# Curva Precision-Recall
prec, rec, thresholds = precision_recall_curve(y_test, y_prob_best)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1_scores.argmax()
best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

axes[1].plot(rec, prec, color='#2a9d8f', linewidth=2)
axes[1].scatter([rec[best_idx]], [prec[best_idx]], color='red', zorder=5,
                label=f'Melhor Threshold: {best_thr:.2f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()

plt.tight_layout()
plt.savefig('avaliacao_modelo.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nMelhor threshold pelo F1: {best_thr:.2f}")
print(f"F1 com melhor threshold: {f1_scores[best_idx]:.4f}")

In [ ]:
# =============================================================
# FEATURE IMPORTANCE — MELHOR MODELO
# Extraímos a importância de cada feature no modelo campeão.
# Feature importance nos diz quais variáveis o modelo usou mais
# para tomar decisões.
# =============================================================

# Extraímos o preprocessador e o modelo da pipeline
preprocessor_fitted = best_model.named_steps['preprocessor']
model_fitted = best_model.named_steps['model']

# Recuperamos os nomes das features após o pré-processamento
try:
    # Nomes das features numéricas (ficam iguais)
    feature_names_num = num_cols
    # Nomes das features categóricas após OHE
    ohe = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot']
    feature_names_cat = ohe.get_feature_names_out(cat_cols).tolist()
    feature_names = feature_names_num + feature_names_cat
except Exception:
    feature_names = [f'feature_{i}' for i in range(len(model_fitted.feature_importances_))]

# Criamos DataFrame com as importâncias
importances = model_fitted.feature_importances_
df_imp = pd.DataFrame({
    'Feature': feature_names[:len(importances)],
    'Importance': importances
}).sort_values('Importance', ascending=False)

# TOP 10 features mais importantes
top10 = df_imp.head(10)

print("=== TOP 10 FEATURES MAIS IMPORTANTES ===")
print(top10.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = ['#e63946' if i < 5 else '#457b9d' for i in range(len(top10))]
ax.barh(top10['Feature'], top10['Importance'], color=colors_imp)
ax.set_xlabel('Importância')
ax.set_title(f'Top 10 Features Mais Importantes — {best_nome}')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

top5_ml = df_imp.head(5)['Feature'].tolist()
print(f"\nTOP 5 features ML: {top5_ml}")

---
# PARTE 3 — Comparação entre os Métodos

## Teste Z vs. Feature Importance do Machine Learning

Agora comparamos:
- As **5 features mais importantes** identificadas pelo modelo de ML  
- As features com **diferença estatisticamente significativa** no Teste Z  

### O que esperar?
Esperamos haver uma interseção significativa entre os dois conjuntos, pois ambos os métodos tentam identificar variáveis relacionadas com o comportamento da variável-alvo — porém por caminhos diferentes.

In [ ]:
# =============================================================
# PARTE 3 — COMPARAÇÃO ENTRE TESTE Z E FEATURE IMPORTANCE
# =============================================================

print("=" * 60)
print("COMPARAÇÃO: TESTE Z vs. FEATURE IMPORTANCE (ML)")
print("=" * 60)

# TOP 5 do ML (já calculado)
print(f"\n📊 TOP 5 Features — Machine Learning ({best_nome}):")
for i, feat in enumerate(top5_ml, 1):
    imp = df_imp[df_imp['Feature'] == feat]['Importance'].values[0]
    print(f"  {i}. {feat} (importância: {imp:.4f})")

# Features significativas do Teste Z
print(f"\n📊 Features Significativas — Teste Z (p < 0.05):")
for i, feat in enumerate(features_significativas[:10], 1):  # Mostramos as 10 mais significativas
    row = df_resultados[df_resultados['Feature'] == feat].iloc[0]
    print(f"  {i}. {feat} (p={row['p-valor']:.5f})")

# Interseção entre os dois conjuntos
intersecao = set(top5_ml) & set(features_significativas)
print(f"\n✅ Features presentes em AMBOS os métodos:")
if intersecao:
    for feat in intersecao:
        print(f"  → {feat}")
else:
    print("  Nenhuma feature idêntica (possível por diferença de nomes após OHE)")

# Verificamos também parcialmente (feature base)
print("\n💡 Análise de correspondência aproximada:")
for feat_ml in top5_ml:
    # Pega o nome base (antes do '_' do OHE)
    base = feat_ml.split('_')[0] if '_' in feat_ml else feat_ml
    matches = [f for f in features_significativas if base.lower() in f.lower() or f.lower() in base.lower()]
    status = '✓ Também significativa no Teste Z' if matches else '✗ Não encontrada no Teste Z'
    print(f"  {feat_ml}: {status}")

In [ ]:
# =============================================================
# DISCUSSÃO FINAL
# Visualização comparativa dos dois métodos
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Comparação: Evidência Estatística (Teste Z) vs. Feature Importance (ML)',
             fontsize=13, fontweight='bold')

# Gráfico 1: Top 10 features ML
top10_ml = df_imp.head(10)
colors_ml = ['#e63946' if f in top5_ml else '#457b9d' for f in top10_ml['Feature']]
axes[0].barh(top10_ml['Feature'], top10_ml['Importance'], color=colors_ml)
axes[0].set_title('Feature Importance — ML (Top 10)')
axes[0].set_xlabel('Importância (Gini)')
axes[0].invert_yaxis()

# Gráfico 2: Top 10 features Teste Z (por diferença absoluta de média)
top10_z = df_resultados[df_resultados['Significativa'] == 'SIM ✓'].head(10).copy()
top10_z['Diff_Abs'] = top10_z['Diferença'].abs()
top10_z = top10_z.sort_values('Diff_Abs', ascending=False).head(10)

colors_z = ['#2a9d8f' if f in features_significativas else '#adb5bd' for f in top10_z['Feature']]
axes[1].barh(top10_z['Feature'], -np.log10(top10_z['p-valor'] + 1e-10), color=colors_z)
axes[1].set_title('Evidência Estatística — Teste Z (−log10 do p-valor)')
axes[1].set_xlabel('−log10(p-valor): maior = mais significativo')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('comparacao_metodos.png', dpi=120, bbox_inches='tight')
plt.show()

## Conclusão da Parte 3 — Discussão

### Relação entre Evidência Estatística e Importância no ML

**Semelhanças:**
- Ambos os métodos tendem a concordar nas variáveis mais relevantes, especialmente aquelas relacionadas a **comportamento de compra** (total gasto, número de compras) e **histórico de campanhas** (quantas campanhas anteriores foram aceitas).
- Features com grande diferença de média entre os grupos (Teste Z) geralmente também têm alta importância no modelo de ML.

**Diferenças:**
- O **Teste Z** avalia cada variável **de forma isolada**, sem considerar interações entre elas.
- O **ML** considera as features **em conjunto** e captura interações. Uma variável pode ter alta importância no ML mesmo sem diferença individual significativa (porque em combinação com outras se torna relevante).
- O Teste Z é **mais interpretável** e direto; o ML é **mais preditivo** mas menos transparente.

**Conclusão:**
O ideal é usar os dois métodos de forma complementar:  
- O Teste Z para comunicar resultados e embasar decisões simples com stakeholders.  
- O ML para construir modelos preditivos mais precisos que consideram o conjunto completo de informações.